# 10.3 Granular synthesis

We can now extract and reassemble frames, but so far the exercise has been been somewhat pointless: worst case we lose information, and best case we get back exactly what we started with. The interesting possibilities open up when we _manipulate_ the frames before reassembling them.

This is the idea behind {vocab}`granular synthesis`: chop a sound into many tiny slices, called {vocab}`grains` (typically tens of milliseconds long), then transform and rearrange those grains to build something new. It is a bit like making a collage out of a photograph, cutting it into little pieces and gluing them back in a new arrangement.

:::{figure}
![Three rows. Top: the source waveform. Middle, labeled extract grains times: a row of six overlapping bell-shaped grains, each a distinct color, covering the source. Bottom, labeled reassemble plus: four of those grains rearranged into a new order with gaps between them.](./assets/fig-granular-collage.png)

Granular synthesis in three steps: extract short grains from the source (each multiplied by a smooth window), then reassemble them, possibly reordered, resized, or otherwise transformed, into a new sound.
:::

Because a grain is so short, it loses much of the recognizable character of the original sound. And a raw grain, sliced out with a hard rectangular window, has abrupt edges that produce an audible click. So in practice we multiply each grain by a smooth window (a Hann window, say) to taper those edges. Here is a handful of 50 ms grains lifted from the running example and played back with a big gap between them, first with hard rectangular edges and then windowed:

:::{audio-list}
{audio}`Raw (rectangular) grains <./assets/audio-grains-rect.wav>`

{audio}`Windowed (Hann) grains <./assets/audio-grains-hann.wav>`

The same grains, played with a rectangular window (note the click at each edge) and with a Hann window (smooth).
:::

## Manipulating grains

Individual grains are not very interesting on their own. The power of granular synthesis comes from manipulating them _as units_ before reassembly. One of the simplest manipulations is to _reorder_ them. We can shuffle grains across the whole signal, or shuffle them only within short segments:

:::{figure}
![Two diagrams, each with a labeled row of grains above a labeled row of output. Top: the grains shuffled into a completely random order. Bottom: the grains shuffled only within blocks of four, marked by vertical dividers, so nearby grains stay roughly together.](./assets/fig-granular-randomize.png)

Two ways to randomize grain order: globally (top), which fully scrambles the sound, or within short segments (bottom), which keeps the large-scale structure while blurring the fine detail.
:::

Reordering grains produces a striking effect. It preserves the overall _texture_ of the sound while erasing its specifics, a kind of controlled blur:

:::{audio-list}
{audio}`Granular texture (grains shuffled within segments) <./assets/audio-granular-texture.wav>`

{audio}`For contrast: the raw samples shuffled <./assets/audio-scrambled-samples.wav>`

Shuffling _grains_ keeps the character of the sound. Shuffling the raw _samples_ (bottom) destroys it entirely, leaving only noise.
:::

That contrast is the whole point. Shuffling grains keeps the sound recognizable, but shuffling the underlying _samples_ (not grains) yields nothing but noise. Working at the level of grains, rather than samples, is what makes the effect musical. Order is not the only property we can manipulate: we could also change the grains' amplitude, duration, pitch, or density before reassembling. You can explore all of these by editing the `manipulate` function below:

In [ ]:
# hide
import numpy as np
import pyquist as pq


def iter_frames(audio, N_H, N_F):
    x = np.asarray(audio.samples).reshape(-1)
    for start in range(0, len(x) - N_F + 1, N_H):
        yield x[start:start + N_F]


def overlap_add(grains, N_H, sample_rate):
    grains = list(grains)
    N_F = len(grains[0])
    out = np.zeros(N_H * (len(grains) - 1) + N_F)
    for k, g in enumerate(grains):
        if len(g) == N_F:
            out[k * N_H:k * N_H + N_F] += g
    return pq.Audio(out.astype(np.float32), sample_rate)


def hann(n):
    return 0.5 * (1 - np.cos(2 * np.pi * np.arange(n) / n))

In [ ]:
# Granular synthesis: chop the sound into overlapping grains, MANIPULATE them,
# and glue them back. Edit `manipulate` to invent your own effect!
def manipulate(grains):
    grains = [g * hann(len(g)) for g in grains]     # smooth each grain's edges
    out = []                                        # shuffle order within blocks
    for i in range(0, len(grains), 100):
        block = grains[i:i + 100]
        np.random.shuffle(block)
        out += block
    return out


N_F = 2048                     # grain size in samples (~46 ms)
N_H = 1024                     # spacing when extracting grains
N_H_out = 1024                 # spacing when reassembling (change to time-stretch!)

audio = pq.Audio.from_file("./assets/audio-trio.wav")
grains = [g for g in iter_frames(audio, N_H, N_F) if len(g) == N_F]
grains = manipulate(grains)
pq.play(overlap_add(grains, N_H_out, audio.sample_rate))

## Time stretching

Here is a particularly useful manipulation. What if we _decouple_ the hop length at which we extract grains from the hop length at which we overlap them back together? Call the extraction hop $N_H$ and the reassembly hop $N_H'$. If $N_H' = 2 N_H$, we spread the grains out to twice their original spacing, doubling the output's duration. If $N_H' = \tfrac{1}{2} N_H$, we pack them together, halving it:

:::{figure}
![Two rows of the same six colored grains. The top row (extract, hop N_H) has the grains at their original spacing. The bottom row (reassemble, hop 2 N_H) has the same grains at double the spacing, so they span twice the width, annotated as twice as long (half speed).](./assets/fig-time-stretch.png)

Time stretching by decoupling the hops. The grains are unchanged, but reassembling them at twice the spacing ($N_H' = 2 N_H$) makes the output twice as long, halving the playback speed.
:::

:::{audio-list}
{audio}`Original <./assets/audio-trio.wav>`

{audio}`Half speed (grains spread out) <./assets/audio-stretch-half.wav>`

{audio}`Double speed (grains packed together) <./assets/audio-stretch-double.wav>`

Granular time stretching. Changing the spacing at reassembly changes the duration, and therefore the playback speed, while the grains themselves are untouched.
:::

We have achieved {vocab}`time stretching`. Spreading or packing the grains changes the total duration, and hence the playback speed, without touching the contents of the grains themselves.

This is the _second_ time we have changed playback speed. The first was {ref}`resampling <sec-resampling>` in [Chapter 7](../ch07/index.md). Listen to the same speed changes done by resampling instead:

:::{audio-list}
{audio}`Half speed via resampling <./assets/audio-resample-half.wav>`

{audio}`Double speed via resampling <./assets/audio-resample-double.wav>`

Resampling also changes the speed, but notice that it changes the _pitch_ too, exactly like slowing down or speeding up a record.
:::

The difference is crucial. Resampling changes duration _and_ pitch together (slower means lower, faster means higher), which was exactly what we wanted for wavetable synthesis. But granular time stretching changes duration while keeping the pitch _constant_. Having both techniques suggests something powerful: _decoupled_ control over pitch and duration. We can first _resample_ the grains to change their pitch, and then independently _time stretch_ them by changing their spacing:

:::{figure}
![Three rows of the same six colored grains. Row 1 (extract, hop N_H): grains at their original size, each containing a slow oscillation. Row 2 (resample, pitch up, shorter): the same grains resampled to be narrower, with a faster oscillation inside, at the same start positions. Row 3 (reassemble, hop 2 N_H): the shorter, higher-pitched grains spread out to double spacing, spanning twice the width.](./assets/fig-decoupled.png)

Decoupled pitch and time. First _resample_ each grain, which shortens it and raises its pitch (row 2). Then _reassemble_ the grains at a wider hop, which stretches the result back out in time (row 3). Because the two steps are independent, the output can be both slower and higher-pitched than the input.
:::

:::{audio-list}
{audio}`Half speed and 20% higher pitch (resample + stretch) <./assets/audio-decoupled.wav>`

Combining resampling (to shift the pitch up 20%) with granular time stretching (to slow to half speed) lets us control the two independently.
:::

In practice, getting a clean result from granular time stretching requires a generous amount of overlap between grains, so that the crossfades between them are smooth.